In [3]:
import pandas as pd
import sys
sys.path.append(r'C:\Users\sssid\OneDrive\Desktop\pharma_supply_chain')
from db_config import get_engine

In [5]:
df = pd.read_csv(r'C:\Users\sssid\OneDrive\Desktop\pharma_supply_chain\data\raw\pharmaceutical_supply_chain.csv')

In [6]:
print(df.shape)
print(df.head(3))
print(df.dtypes)

(100000, 4)
         Drug  Demand_Forecast  Optimal_Stock_Level Restocking_Strategy
0   Metformin             7750                 4753             Monthly
1  Lisinopril             5136                 9965           Quarterly
2   Metformin             3183                 2933             Monthly
Drug                   object
Demand_Forecast         int64
Optimal_Stock_Level     int64
Restocking_Strategy    object
dtype: object


In [7]:
engine = get_engine()
df.to_sql('supply_chain', con=engine,
          if_exists='replace',   # replace if already exists
          index=False,
          chunksize=1000)

print(f"Loaded {len(df)} rows into MySQL table: supply_chain")

Loaded 100000 rows into MySQL table: supply_chain


In [8]:
sales = pd.read_csv(r'C:\Users\sssid\OneDrive\Desktop\pharma_supply_chain\data\raw\salesdaily.csv')

In [9]:
# The pharma sales dataset has columns like: datum, M01AB, M01AE...
# Melt from wide to long format for MySQL
sales['datum'] = pd.to_datetime(sales['datum'])
sales_long = sales.melt(
    id_vars='datum',
    var_name='drug_code',
    value_name='units_sold'
)
sales_long.rename(columns={'datum': 'sale_date'}, inplace=True)

sales_long.to_sql('pharma_sales', con=engine,
                  if_exists='replace', index=False)

print(f"Loaded {len(sales_long)} rows into pharma_sales")

Loaded 25272 rows into pharma_sales
